In [49]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [50]:
print("CUDA available:", torch.cuda.is_available())
print("Torch CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
Torch CUDA version: 11.8
GPU: NVIDIA GeForce GTX 1650


In [51]:
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from torchvision.transforms import v2

In [52]:
from datasets import load_dataset

ds = load_dataset("AI-Lab-Makerere/beans")

In [53]:
df_train = pd.DataFrame(ds['train'])
df_test = pd.DataFrame(ds['test'])

In [54]:
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 1034 entries, 0 to 1033
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   image_file_path  1034 non-null   str   
 1   image            1034 non-null   object
 2   labels           1034 non-null   int64 
dtypes: int64(1), object(1), str(1)
memory usage: 194.7+ KB


In [55]:
df_train = df_train.drop(columns='image_file_path')
df_test = df_test.drop(columns='image_file_path')

In [56]:
df_train.head(1)

,image,labels
0,<PIL.JpegImagePlugin.JpegImageFile image mode=...,0


In [57]:
import torchvision.models as models

In [58]:
all_models = models.list_models()
all_models

['alexnet',
 'convnext_base',
 'convnext_large',
 'convnext_small',
 'convnext_tiny',
 'deeplabv3_mobilenet_v3_large',
 'deeplabv3_resnet101',
 'deeplabv3_resnet50',
 'densenet121',
 'densenet161',
 'densenet169',
 'densenet201',
 'efficientnet_b0',
 'efficientnet_b1',
 'efficientnet_b2',
 'efficientnet_b3',
 'efficientnet_b4',
 'efficientnet_b5',
 'efficientnet_b6',
 'efficientnet_b7',
 'efficientnet_v2_l',
 'efficientnet_v2_m',
 'efficientnet_v2_s',
 'fasterrcnn_mobilenet_v3_large_320_fpn',
 'fasterrcnn_mobilenet_v3_large_fpn',
 'fasterrcnn_resnet50_fpn',
 'fasterrcnn_resnet50_fpn_v2',
 'fcn_resnet101',
 'fcn_resnet50',
 'fcos_resnet50_fpn',
 'googlenet',
 'inception_v3',
 'keypointrcnn_resnet50_fpn',
 'lraspp_mobilenet_v3_large',
 'maskrcnn_resnet50_fpn',
 'maskrcnn_resnet50_fpn_v2',
 'maxvit_t',
 'mc3_18',
 'mnasnet0_5',
 'mnasnet0_75',
 'mnasnet1_0',
 'mnasnet1_3',
 'mobilenet_v2',
 'mobilenet_v3_large',
 'mobilenet_v3_small',
 'mvit_v1_b',
 'mvit_v2_s',
 'quantized_googlenet',
 '

In [59]:
weights_enum = models.get_model_weights("resnet50")
print(list(weights_enum)) 

[ResNet50_Weights.IMAGENET1K_V1, ResNet50_Weights.IMAGENET1K_V2]


Loaded a Pretrained model below

In [60]:
model = models.get_model(name="resnet50",weights="ResNet50_Weights.IMAGENET1K_V2")

In [61]:
for params in model.parameters():
    params.requires_grad = False

In [62]:
print(model.fc)

Linear(in_features=2048, out_features=1000, bias=True)


In [63]:
weights = models.ResNet50_Weights.IMAGENET1K_V2
transform_metrics = weights.transforms()
transform_metrics

ImageClassification(
    crop_size=[224]
    resize_size=[232]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)

In [64]:
class datasetAugmentTrain(Dataset):
    def __init__(self,data,metrics):
        super().__init__()
        self.data = data
        self.crop_size = metrics.crop_size
        self.resize_size = metrics.resize_size
        self.mean = metrics.mean
        self.std = metrics.std
        self.preprocess= v2.Compose([
            v2.Resize(self.resize_size),
            v2.RandomCrop(self.crop_size),
            v2.RandomHorizontalFlip(p=0.5),
            v2.ColorJitter(brightness=0.2, contrast=0.2,saturation=0.3),
            v2.RandomErasing(p=0.25),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=self.mean,std=self.std)
        ])
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        data = self.data.iloc[index]
        return self.preprocess(data["image"]), torch.tensor(data["labels"])

In [65]:
class datasetAugmentTest(Dataset):
    def __init__(self,data,metrics):
        super().__init__()
        self.data = data
        self.crop_size = metrics.crop_size
        self.resize_size = metrics.resize_size
        self.mean = metrics.mean
        self.std = metrics.std
        self.preprocess= v2.Compose([
            v2.Resize(self.resize_size),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=self.mean,std=self.std)
        ])
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        data = self.data.iloc[index]
        return self.preprocess(data["image"]), torch.tensor(data["labels"])

In [66]:
class Model(nn.Module):
    def __init__(self, classes):
        super().__init__()

        # model for transfer learning
        self.model = models.get_model(name="resnet50",weights="ResNet50_Weights.IMAGENET1K_V2")

        # FC layers in_features and creation of new FC_layers
        in_features = self.model.fc.in_features
        self.model.fc = nn.Sequential(
            nn.Linear(in_features=in_features,out_features=512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(in_features=512,out_features=classes),
        )

        # Freeze all Layers Parameters
        for params in self.model.parameters():
            params.requires_grad = False
        
        # UnFreeze only FC Layers Parameters
        for params in self.model.fc.parameters():
            params.requires_grad = True
    
    def forward(self,x):
        return self.model(x)

In [67]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [68]:
import torch.optim as optim
model = Model(3).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(lr=0.0001,params=model.parameters())

In [69]:
train_loader = DataLoader(datasetAugmentTrain(df_train,transform_metrics),batch_size=32,shuffle=True)
test_loader = DataLoader(datasetAugmentTrain(df_test,transform_metrics),batch_size=32,shuffle=False)

In [70]:
def train_model(model, train_loader, loss_fn, optimizer, epochs, device):
    for epoch in range(epochs):
        model.train()
        total_loss_per_epoch = 0 
        for batch_features,batch_labels in train_loader:
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

            #Forward pass
            y_hat = model(batch_features)
            loss = loss_fn(y_hat, batch_labels)

            #backward pass
            optimizer.zero_grad()
            loss.backward()

            #update wieghts
            optimizer.step()
            total_loss_per_epoch += loss.item()
        
        avg_loss = total_loss_per_epoch / len(train_loader)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

if __name__ == '__main__':
    train_model(model, train_loader, loss_fn, optimizer, epochs=50, device=device)

Epoch 1/50, Loss: 1.0306
Epoch 2/50, Loss: 0.8580
Epoch 3/50, Loss: 0.6804
Epoch 4/50, Loss: 0.5676
Epoch 5/50, Loss: 0.5013
Epoch 6/50, Loss: 0.4731
Epoch 7/50, Loss: 0.4296
Epoch 8/50, Loss: 0.3942
Epoch 9/50, Loss: 0.3754
Epoch 10/50, Loss: 0.3396
Epoch 11/50, Loss: 0.3283
Epoch 12/50, Loss: 0.2978
Epoch 13/50, Loss: 0.2948
Epoch 14/50, Loss: 0.2818
Epoch 15/50, Loss: 0.2689
Epoch 16/50, Loss: 0.2578
Epoch 17/50, Loss: 0.2293
Epoch 18/50, Loss: 0.2536
Epoch 19/50, Loss: 0.2448
Epoch 20/50, Loss: 0.2236
Epoch 21/50, Loss: 0.2369
Epoch 22/50, Loss: 0.2228
Epoch 23/50, Loss: 0.2099
Epoch 24/50, Loss: 0.2155
Epoch 25/50, Loss: 0.2092
Epoch 26/50, Loss: 0.1893
Epoch 27/50, Loss: 0.2109
Epoch 28/50, Loss: 0.1900
Epoch 29/50, Loss: 0.1999
Epoch 30/50, Loss: 0.1738
Epoch 31/50, Loss: 0.1756
Epoch 32/50, Loss: 0.1641
Epoch 33/50, Loss: 0.1587
Epoch 34/50, Loss: 0.1643
Epoch 35/50, Loss: 0.1873
Epoch 36/50, Loss: 0.1562
Epoch 37/50, Loss: 0.1504
Epoch 38/50, Loss: 0.1498
Epoch 39/50, Loss: 0.

In [71]:
torch.save(model.state_dict(), "resnet50_transfer_learning_leaf_health_multi_classification3.pth")

In [73]:
model.eval()
total=0
correct =0 
with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)
        
        # Get predictions
        outputs = model(batch_features)
        _, predicted = torch.max(outputs, dim=1)
        
        # Count accuracy
        total += batch_labels.size(0)
        correct += (predicted == batch_labels).sum().item()
print("Test accuracy:", correct / total)

Test accuracy: 0.9140625


In [79]:
# 1. Get the image and its REAL label from the dataset
image, actual_label = test_loader.dataset[105]

# 2. Predict
model.eval()
with torch.no_grad():
    output = model(image.unsqueeze(0).to(device))
    _, pred = torch.max(output, 1)

print(f"Prediction: {pred.item()}")
print(f"Actual Label from Dataset: {actual_label}")

if pred.item() == actual_label:
    print("The model is actually correct for this index!")
else:
    print("There is a genuine mismatch for this specific image.")

Prediction: 2
Actual Label from Dataset: 2
The model is actually correct for this index!
